In [11]:
import pandas as pd
import numpy as np

data_path = "/Users/zacharyfrederick/Downloads/Historical_returns.csv" 



In [12]:
df = pd.read_csv(data_path, skiprows=-1).transpose()
df.head()

,0
Time-weighted rate of return (pre-tax),Individual Z27309283
Feb 2026,+1.18%
Jan 2026,+1.98%
Dec 2025,+0.29%
Nov 2025,-7.70%


In [ ]:
# Drop header row if still present (first row after transpose), then keep only valid date rows
df = df.iloc[1:].copy() if df.index[0] == "Time-weighted rate of return (pre-tax)" else df.copy()
df.columns = ["return"]
# Parse percentage strings to numeric (e.g. "+1.18%" -> 0.0118)
df["return"] = (
    df["return"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace("+", "", regex=False)
    .str.strip()
)
df["return"] = pd.to_numeric(df["return"], errors="coerce") / 100
df = df.dropna(subset=["return"])  # drops "Unnamed: 39" and any other invalid rows
df.index = pd.to_datetime(df.index, format="%b %Y")
df = df.sort_index(ascending=True)

df.head(10)

,return
2023-01-01,0.0010
2023-02-01,-0.0038
2023-03-01,0.0086
2023-04-01,-0.0163
2023-05-01,-0.0304
2023-06-01,0.1759
2023-07-01,0.0391
2023-08-01,-0.0545
2023-09-01,-0.0800
2023-10-01,-0.0476


In [14]:
# Portfolio metrics (monthly returns -> annualized where applicable)
MONTHS_PER_YEAR = 12
rf_monthly = 0.02 / MONTHS_PER_YEAR  # risk-free approx

r = df["return"]
n = len(r)

# Volatility (annualized)
vol_annual = r.std() * np.sqrt(MONTHS_PER_YEAR)

# Sharpe ratio (annualized)
sharpe = (r.mean() - rf_monthly) / r.std() * np.sqrt(MONTHS_PER_YEAR) if r.std() > 0 else np.nan

# Cumulative wealth and drawdowns
wealth = (1 + r).cumprod()
running_max = wealth.cummax()
drawdown = (wealth - running_max) / running_max
max_dd = drawdown.min()
max_dd_duration = (drawdown < 0).astype(int).groupby((drawdown >= 0).cumsum()).sum().max()

# Calmar (annualized return / max drawdown)
ann_return = (wealth.iloc[-1] ** (MONTHS_PER_YEAR / n) - 1) if n > 0 else np.nan
calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

# Sortino (downside deviation; use same geometric ann. return as Calmar)
downside_returns = r[r < 0]
downside_std = downside_returns.std() * np.sqrt(MONTHS_PER_YEAR) if len(downside_returns) > 0 else np.nan
sortino = (ann_return - 0.02) / downside_std if (downside_std and downside_std > 0) else np.nan

# Summary table
metrics = pd.DataFrame({
    "Volatility (ann.)": [f"{vol_annual:.2%}"],
    "Sharpe ratio": [round(sharpe, 2)],
    "Sortino ratio": [round(sortino, 2)],
    "Max drawdown": [f"{max_dd:.2%}"],
    "Calmar ratio": [round(calmar, 2)],
    "Ann. return": [f"{ann_return:.2%}"],
})
metrics

,Volatility (ann.),Sharpe ratio,Sortino ratio,Max drawdown,Calmar ratio,Ann. return
0,22.55%,1.03,2.28,-17.15%,1.49,25.52%


In [15]:
df

,return
2023-01-01,0.0010
2023-02-01,-0.0038
2023-03-01,0.0086
2023-04-01,-0.0163
2023-05-01,-0.0304
2023-06-01,0.1759
2023-07-01,0.0391
2023-08-01,-0.0545
2023-09-01,-0.0800
2023-10-01,-0.0476
